# 3. Harness Design — Deep Dive

Continuing from [`02_Evaluation_and_Results.ipynb`](02_Evaluation_and_Results.ipynb). Read time: ~12 minutes. No API key required by default — this notebook reads the bundled audit-trail dump and re-counts dangling references using only the bundled JSON.

An LLM-powered system has the same trust problem no matter how big the model. RLHF, chain-of-thought, and bigger context windows all push trust toward the model. This project pushes it elsewhere — into a mechanical audit substrate the reviewer can navigate themselves. The four harnesses are how that substrate gets enforced.

Every claim in the final report ties back, by id, to a tool observation. A reviewer doesn't ask the model "why did you say that?" — they query the audit trail.

## The four harnesses

A *harness* is a cross-cutting enforcement layer that wraps the agent graph. Each is narrow and one-job:

| Harness | Job | When it runs |
| :--- | :--- | :--- |
| **Input** | Validates the incoming review trigger and bundle shape (tier schemas, timestamp continuity, Terraform parseability). Rejects garbage before any specialist wastes a reasoning cycle on it. | At cycle start |
| **Reasoning** | Validates the structural honesty of the reasoning chain: every cited `evidence_ref` resolves to a real audit row in the same cycle (no missing / dangling / foreign-cycle refs); `finding_type` is in the allowed set; a `tight` drift verdict must be backed by at least one substantive observation (rejects "all-empty observations = tight"). The *semantic* narrative-vs-evidence drift-check itself is the Cross-Tier Evaluator's job; the Reasoning Harness validates that the Evaluator's drift output is structurally honest. | After each specialist returns and at synthesis |
| **Action** | Gates the synthesized recommendation: well-formedness against the schema, and traceability (every cited `evidence_ref` resolves to a real audit row; zero dangling). | Before the recommendation is returned |
| **Orchestration** | Cycle-end consistency checks at the graph level: every dispatched specialist returned, no orphan substance rows, every harness event links to its audit row. | At cycle_complete |

All harness events are persisted to a separate SQLite table (`harness_trail`) keyed to the same `cycle_id` as the substance records in `audit_records`. The exact check list per harness lives in [`docs/harnesses.md`](../docs/harnesses.md); the code is in [`src/harnesses/`](../src/harnesses/). The table above stays conservative and names only what's implemented today — the Reasoning Harness row in particular distinguishes between the *implemented* check (ref-resolution and drift-output structural honesty) and the *semantic* drift judgment, which the Cross-Tier Evaluator performs and the Reasoning Harness then validates.

## A recording — scenario 08

Load the bundled audit-trail dump and see what one full cycle's harness output looks like.

In [1]:
import json
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
TRACE_PATH = PROJECT_ROOT / 'sample_runs' / 'traces' / 'scenario_08_trace.json'

trace = json.loads(TRACE_PATH.read_text())

print(f"Loaded cycle: {trace['cycle_id']}")
print(f"Application : {trace['application_id']}")
print()
print('--- traceability summary (Action Harness gate output) ---')
for k, v in trace['summary'].items():
    print(f'  {k:24s}: {v}')
print()
print("Read: 'unique_refs_cited' is how many distinct audit rows the recommendation")
print("      claims as evidence. 'resolved' is how many actually exist in the audit DB.")
print("      'dangling' MUST be zero — the Action Harness refuses to surface a")
print("      recommendation with dangling refs.")

Loaded cycle: cycle_20260604_150610_37995130
Application : app-08

--- traceability summary (Action Harness gate output) ---
  unique_refs_cited       : 26
  resolved                : 26
  dangling                : 0
  observation_refs        : 24
  specialist_finding_refs : 2

Read: 'unique_refs_cited' is how many distinct audit rows the recommendation
      claims as evidence. 'resolved' is how many actually exist in the audit DB.
      'dangling' MUST be zero — the Action Harness refuses to surface a
      recommendation with dangling refs.


## Specialists that fired in this cycle

Three tier specialists are available — Compute Analyst, Data-Layer Analyst, Network Analyst. The **System Mapper** parses Terraform to detect which tiers are actually present in the topology, and the **Supervisor** dispatches only the specialists whose tiers are there. Running a Network Analyst against an infrastructure with no network tier would produce a no-issue-found finding with empty observations — wasteful and noisy.

Scenario 08's infrastructure has compute + database (no cache, no separate network tier), so two specialists fire below. That's the expected behavior, not a missing piece. Each produces a finding with confidence + reasoning + cited evidence.

In [2]:
for sf in trace['specialist_findings']:
    finding = sf['content']
    print(f"=== {sf['agent']} (record id={sf['id']}, ts={sf['timestamp']}) ===")
    print(f"  finding_type    : {finding.get('finding_type')}")
    print(f"  primary_tier    : {finding.get('primary_tier')}")
    print(f"  confidence      : {finding.get('confidence')}")
    headline = finding.get('headline', '') or ''
    if len(headline) > 240:
        # Trim at the last word boundary before 240 to avoid mid-word cuts.
        cut = headline.rfind(' ', 0, 240)
        headline = headline[:cut] + '…' if cut > 0 else headline[:240] + '…'
    print(f'  headline        : {headline}')
    summary = finding.get('reasoning_summary') or ''
    print('  reasoning_summary (first 240 chars):')
    print(f'    {summary[:240]}')
    print()

=== compute_analyst (record id=451, ts=2026-06-04 15:07:58) ===
  finding_type    : issue_found
  primary_tier    : compute
  confidence      : 0.97
  headline        : Severe P95 latency SLA breach during weekday business hours: latency reaches 620-670ms against a 300ms SLA target on a tier-1 checkout service.
  reasoning_summary (first 240 chars):
    The application_p95_latency_ms shows a strongly bimodal distribution: 781 records cluster at 120-175ms (off-peak) while 335 records cluster at 618-674ms (weekday business hours 9am-6pm). 370 of 1344 total records (27.5%) breach the 300ms SL

=== data_layer_analyst (record id=460, ts=2026-06-04 15:09:20) ===
  finding_type    : issue_found
  primary_tier    : database
  confidence      : 0.97
  headline        : Database has 6 slow queries causing severe SLA breaches (p95 ~791ms vs 300ms target) during weekday business hours, with only 1 replica and no R/W splitting to distribute read load.
  reasoning_summary (first 240 chars):
    The 

## Drift-check verdicts from the Cross-Tier Evaluator

At synthesis time, the **Cross-Tier Evaluator** runs a drift-check on each specialist's narrative against its cited evidence. The verdicts are persisted to the recommendation row and shown below. The implemented verdict vocabulary is `tight` / `loose` / `contradictory`: `tight` means narrative and evidence agree; `loose` flags the finding for the human reviewer; `contradictory` flags a narrative-vs-evidence inversion that prevents the finding from carrying weight in the synthesis.

The **Reasoning Harness** then validates that the Evaluator's drift output is structurally honest — every cited `supporting_evidence_ref` resolves to a real audit row in this cycle, and a `tight` verdict isn't built on empty observations. That validation is what makes the verdicts below trustworthy as data; the semantic judgment itself is the Evaluator's.

In [3]:
reconciliation = trace['recommendation'].get('reconciliation') or {}
drift_checks   = reconciliation.get('drift_check') or []

if drift_checks:
    print('--- Cross-Tier Evaluator drift-check verdicts ---')
    for dc in drift_checks:
        verdict   = dc.get('verdict', '?')
        agent     = dc.get('agent', '?')
        # The narrative field holds the evaluator's free-text rationale.
        narrative = (dc.get('narrative') or '')[:200]
        print(f'  {agent:24s} -> {verdict}')
        if narrative:
            print(f'    {narrative}')
        print()
else:
    print('(no drift_check block in this trace)')

--- Cross-Tier Evaluator drift-check verdicts ---
  compute_analyst          -> loose
    The Compute Analyst correctly identified the SLA breach pattern (evidence_refs 440, 442, 446) and the bimodal latency distribution, but attributed it to compute insufficiency and recommended scaling-p

  data_layer_analyst       -> tight
    The Data Layer Analyst's conclusion that the database is the root cause is strongly supported by the cited evidence. The db_query_p95_latency_ms time pattern (evidence_ref 455) shows the same weekday 



## Every claim is anchored — the backward walk

Pick one cited evidence reference and walk it backward to the row that produced it. This is the project's traceability proof reduced to one chain. A reviewer doing a spot-check would do exactly this.

In [4]:
evidence_chain = trace['evidence_chain']  # dict keyed by audit_record id (as string)

# Prefer an evidence ref cited by a TIGHT-verdict finding for a clean story:
# walking evidence from a finding the Evaluator endorsed reads better than
# walking from one it flagged loose. Fall back to lowest-id ref if no tight
# verdicts (e.g., no-issue cycles).
tight_refs: set[str] = set()
for dc in trace['recommendation'].get('reconciliation', {}).get('drift_check', []):
    if dc.get('verdict') == 'tight':
        tight_refs.update(str(r) for r in (dc.get('supporting_evidence_refs') or []))
candidates = [k for k in evidence_chain if k in tight_refs] or list(evidence_chain.keys())
first_ref_key = sorted(candidates, key=lambda k: int(k))[0]
observation = evidence_chain[first_ref_key]

print(f'=== Step A: the observation (audit row id={first_ref_key}) ===')
print(f"  type      : {observation.get('type')}")
print(f"  agent     : {observation.get('agent')}")
print(f"  category  : {observation.get('category')}")
print(f"  status    : {observation.get('status')}    (resolved = the chain ended here OK)")
print(f"  parent_id : {observation.get('parent_id')}")
print()
print(f"  cited_by  : {observation.get('cited_by')}")
print()

# The observation's content describes the underlying tool call (tool_name + result).
tool_content = observation.get('content', {})
tool_name = tool_content.get('tool_name')
if tool_name:
    print('=== Step B: the tool call that produced this observation ===')
    print(f'  tool_name : {tool_name}')
    result_preview = json.dumps(tool_content.get('result', {}), indent=2)[:400]
    print('  result (first 400 chars):')
    print(result_preview)

print()
print('That is the chain: recommendation -> evidence_ref -> observation -> tool call -> agent.')
print('Every claim in the rendered report can be walked this way.')

=== Step A: the observation (audit row id=428) ===
  type      : observation
  agent     : data_layer_analyst
  category  : evidence
  status    : resolved    (resolved = the chain ended here OK)
  parent_id : 427

  cited_by  : ['reconciliation.specialist_findings_summary[1].evidence_refs', 'reconciliation.drift_check[1].supporting_evidence_refs']

=== Step B: the tool call that produced this observation ===
  tool_name : get_summary_statistics
  result (first 400 chars):
{
  "app_name": "app-08",
  "tier": "database",
  "metric": "db_query_p95_latency_ms",
  "statistics": {
    "mean": 274.3518601190476,
    "p50": 131.7,
    "p90": 743.98,
    "p95": 790.8849999999999
  }
}

That is the chain: recommendation -> evidence_ref -> observation -> tool call -> agent.
Every claim in the rendered report can be walked this way.


## The Orchestration Harness — cycle-end consistency

After the recommendation is produced, the Orchestration Harness runs cycle-end checks at the graph level rather than at any single agent. The categories it covers, per [`docs/harnesses.md`](../docs/harnesses.md):

- **All dispatched specialists returned.** If the Supervisor dispatched Compute + Data-Layer, both must have completed.
- **No orphan substance rows.** Every audit row has a valid parent or is a top-level cycle event.
- **Every harness event is linked.** Every row in `harness_trail` either points to its target row in `audit_records` (via `related_event_id`) or carries `related_event_id=NULL` for cycle-level checks.
- **Evidence-ref density.** A no-action `finding_type` is allowed to cite zero refs; an action-claiming `finding_type` must cite at least one.

Orchestration Harness failures are rare in practice but they're how the system catches subtle graph-shape bugs that no individual agent can see. The exact implemented checks are in [`src/harnesses/orchestration.py`](../src/harnesses/orchestration.py).

## Mechanical chain verification

The summary block above showed `dangling: 0`. The cell below re-counts the chain entries using only the bundled JSON — internal consistency of the trace dump. For a stronger verification (re-walk against the original audit DB), the repo ships [`scripts/verify_evidence_chain.sh`](../scripts/verify_evidence_chain.sh), which the integration test calls. That script needs a live `.audit_db` to run against; the in-notebook version below works offline.

What the cell proves: every entry in `evidence_chain` carries `status: 'resolved'`, and the recount matches the trace's reported `dangling: 0`. If even one entry had `status: 'dangling'`, we'd see a non-zero count.

In [5]:
chain = trace['evidence_chain']
chain_ids = {int(k) for k in chain.keys()}

unresolved = [(rid, row.get('status'))
              for rid, row in chain.items()
              if row.get('status') != 'resolved']

reported = trace['summary']
actual_unique   = len(chain_ids)
actual_dangling = len(unresolved)

print(f"trace summary said : unique_refs_cited={reported.get('unique_refs_cited')}, "
      f"resolved={reported.get('resolved')}, dangling={reported.get('dangling')}")
print(f"recounted here     : unique_refs_cited={actual_unique}, "
      f"resolved={actual_unique - actual_dangling}, dangling={actual_dangling}")
print()
if actual_dangling == 0 and reported.get('dangling') == 0:
    print('VERIFIED: trace summary matches recount; every cited evidence_ref carries status=resolved.')
else:
    print(f'MISMATCH: {actual_dangling} unresolved row(s): {unresolved[:5]}')

trace summary said : unique_refs_cited=26, resolved=26, dangling=0
recounted here     : unique_refs_cited=26, resolved=26, dangling=0

VERIFIED: trace summary matches recount; every cited evidence_ref carries status=resolved.


## Optional: run a live cycle yourself

Everything above ran from the bundled artifact. To produce one of these traces yourself with a real LLM cycle, the easiest path is the wrapper:

```bash
make scenario APP=app-08    # runs agents + renders report + scores against gold
                            # ~5-10 min, ~$0.10 with Haiku; more with Opus
```

Or the Docker variant for a hermetic environment:

```bash
docker compose up --build live-llm
```

Both require `ANTHROPIC_API_KEY` in `.env`. After either finishes, `./demo-output/report.md` carries the rendered report and the audit DB at `.audit_db/audit.db` holds the full trail. Full instructions in [`docs/running.md`](../docs/running.md).

## Why these four harnesses, in this order

1. **The audit trail is a foreign-key tree, not a vector store.** A human navigates it with one SQL query — no similarity search, no LLM-in-the-loop interpretation. That's what makes traceability mechanical rather than aspirational.
2. **Substance and enforcement live in separate tables.** `audit_records` is the substance (tool calls, observations, findings, recommendations); `harness_trail` is the enforcement (gate verdicts, drift checks, orchestration checks). They share `cycle_id` and link by id. Separating them means a Reasoning Harness verdict can never silently overwrite a specialist's finding — the substance row stays exactly as the specialist wrote it; the harness's opinion is its own row.
3. **Four harnesses cover four distinct failure modes.** Bad input (Input), reasoning that doesn't match its evidence (Reasoning), recommendations that don't meet the quality bar (Action), graph-level inconsistency (Orchestration). Each is narrow. Adding a fifth would be a code-review conversation, not an architectural one.
4. **The Action Harness gates on traceability before content quality.** A recommendation with dangling evidence_refs fails the gate even if the prose is brilliant. Structure first, content second — that ordering is what makes the system safe to surface to a human reviewer.

## End of the tour

You've now seen all three artifacts: what the system does ([01](01_Architecture_Overview.ipynb)), how it's measured ([02](02_Evaluation_and_Results.ipynb)), how correctness is enforced (this notebook). Going further:

- Design narrative: [`ARCHITECTURE.md`](../ARCHITECTURE.md) and [`docs/decisions.md`](../docs/decisions.md) — both lead with constraints and explain the trade-offs rejected.
- Harness model: [`docs/harnesses.md`](../docs/harnesses.md). Audit substrate: [`docs/audit-trail.md`](../docs/audit-trail.md).
- Live experience: [`docs/running.md`](../docs/running.md) covers three paths from "thirty-second hermetic Docker demo" to "clone, install, run all 18 scenarios."